<a href="https://colab.research.google.com/github/Matth-01/puc-pocos-comp-ds-012026-aura/blob/main/C%C3%B3pia_de_Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AURA - Predição de Aprovação de Empréstimos Bancários

PUC Minas - Ciência da Computação
Data Science e Big Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report
)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train_u6lujuX_CVtuZ9i.csv to train_u6lujuX_CVtuZ9i.csv


In [ ]:
df = pd.read_csv('train_u6lujuX_CVtuZ9i.csv')

print("Registros:", df.shape[0])
print("Atributos:", df.shape[1])

df.head()

Registros: 367
Atributos: 12


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,LP001015,Male,Yes,0,Graduate,No,5720,0,110.0,360.0,1.0,Urban
1,LP001022,Male,Yes,1,Graduate,No,3076,1500,126.0,360.0,1.0,Urban
2,LP001031,Male,Yes,2,Graduate,No,5000,1800,208.0,360.0,1.0,Urban
3,LP001035,Male,Yes,2,Graduate,No,2340,2546,100.0,360.0,NaN,Urban
4,LP001051,Male,No,0,Not Graduate,No,3276,0,78.0,360.0,1.0,Urban


# Dataset Utilizado

Nome: Loan Prediction Problem Dataset

Fonte: Kaggle

Objetivo:
Prever a aprovação ou reprovação de empréstimos bancários.

Variável-alvo:
Loan_Status

Tipo do problema:
Classificação Binária

# Análise Estatística

In [ ]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,367.000000,367.000000,362.000000,361.000000,338.000000
mean,4805.599455,1569.577657,136.132597,342.537396,0.825444
std,4910.685399,2334.232099,61.366652,65.156643,0.380150
min,0.000000,0.000000,28.000000,6.000000,0.000000
25%,2864.000000,0.000000,100.250000,360.000000,1.000000
50%,3786.000000,1025.000000,125.000000,360.000000,1.000000
75%,5060.000000,2430.500000,158.000000,360.000000,1.000000
max,72529.000000,24000.000000,550.000000,480.000000,1.000000


In [ ]:
colunas_numericas = [
    'ApplicantIncome',
    'CoapplicantIncome',
    'LoanAmount',
    'Loan_Amount_Term'
]

for coluna in colunas_numericas:
    print(f'\n{coluna}')
    print("Média:", df[coluna].mean())
    print("Mediana:", df[coluna].median())
    print("Mínimo:", df[coluna].min())
    print("Máximo:", df[coluna].max())
    print("Desvio Padrão:", df[coluna].std())


ApplicantIncome
Média: 4805.599455040872
Mediana: 3786.0
Mínimo: 0
Máximo: 72529
Desvio Padrão: 4910.685398980398

CoapplicantIncome
Média: 1569.5776566757493
Mediana: 1025.0
Mínimo: 0
Máximo: 24000
Desvio Padrão: 2334.2320986863465

LoanAmount
Média: 136.13259668508286
Mediana: 125.0
Mínimo: 28.0
Máximo: 550.0
Desvio Padrão: 61.366652393018214

Loan_Amount_Term
Média: 342.53739612188366
Mediana: 360.0
Mínimo: 6.0
Máximo: 480.0
Desvio Padrão: 65.1566434139972


## Distribuição da variável-alvo

In [ ]:
sns.countplot(data=df, x='Loan_Status')
plt.title('Distribuição das Aprovações')
plt.show()

ValueError: Could not interpret value `Loan_Status` for `x`. An entry with this name does not appear in `data`.

Observa-se que a maioria dos empréstimos presentes no dataset foi aprovada, indicando um leve desbalanceamento entre as classes.

# EDA
# Histórico de Crédito

In [ ]:
sns.countplot(
    data=df,
    x='Credit_History',
    hue='Loan_Status'
)

plt.show()


#Área do Imóvel

In [ ]:
sns.countplot(
    data=df,
    x='Property_Area',
    hue='Loan_Status'
)

plt.show()

# Distribuição da Renda

In [ ]:
sns.histplot(df['ApplicantIncome'], bins=30)
plt.show()

# Pré-processamento dos Dados

In [ ]:
df_modelo = df.copy()

df_modelo.drop('Loan_ID', axis=1, inplace=True)

df_modelo['Dependents'] = df_modelo['Dependents'].replace('3+', 3)

cat_cols = [
    'Gender',
    'Married',
    'Education',
    'Self_Employed',
    'Property_Area'
]

for col in cat_cols:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].mode()[0])

num_cols = [
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

for col in num_cols:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())

In [ ]:
df_modelo['TotalIncome'] = (
    df_modelo['ApplicantIncome']
    + df_modelo['CoapplicantIncome']
)

df_modelo['EMI'] = (
    df_modelo['LoanAmount']
    / df_modelo['Loan_Amount_Term']
)

In [ ]:
df_modelo['Dependents'] = df_modelo['Dependents'].replace('3+', '3')

for col in df_modelo.columns:
    if df_modelo[col].dtype == 'object':
        df_modelo[col] = df_modelo[col].fillna('Desconhecido')
        df_modelo[col] = df_modelo[col].astype(str)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df_modelo.columns:
    if df_modelo[col].dtype == 'object':
        df_modelo[col] = le.fit_transform(df_modelo[col])

In [ ]:
X = df_modelo.drop('Loan_Status', axis=1)
y = df_modelo['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Modelo Base - Random Forest

In [ ]:
modelo_base = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

modelo_base.fit(X_train, y_train)

pred = modelo_base.predict(X_test)

In [ ]:
print("Acurácia:", accuracy_score(y_test,pred))
print("Precisão:", precision_score(y_test,pred))
print("Recall:", recall_score(y_test,pred))
print("F1:", f1_score(y_test,pred))

# Experimento 1

Hipótese:
Aumentar o número de árvores pode melhorar a generalização do modelo.

In [ ]:
modelo_exp1 = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

modelo_exp1.fit(X_train, y_train)

pred_exp1 = modelo_exp1.predict(X_test)

accuracy_exp1 = accuracy_score(y_test, pred_exp1)
precision_exp1 = precision_score(y_test, pred_exp1)
recall_exp1 = recall_score(y_test, pred_exp1)
f1_exp1 = f1_score(y_test, pred_exp1)

print("Acurácia (Experimento 1):", accuracy_exp1)
print("Precisão (Experimento 1):", precision_exp1)
print("Recall (Experimento 1):", recall_exp1)
print("F1 (Experimento 1):", f1_exp1)

# Experimento 2

Hipótese:
Limitar a profundidade reduz overfitting.

In [ ]:
modelo_exp2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

modelo_exp2.fit(X_train, y_train)

pred_exp2 = modelo_exp2.predict(X_test)

accuracy_exp2 = accuracy_score(y_test, pred_exp2)
precision_exp2 = precision_score(y_test, pred_exp2)
recall_exp2 = recall_score(y_test, pred_exp2)
f1_exp2 = f1_score(y_test, pred_exp2)

print("Acurácia (Experimento 2):", accuracy_exp2)
print("Precisão (Experimento 2):", precision_exp2)
print("Recall (Experimento 2):", recall_exp2)
print("F1 (Experimento 2):", f1_exp2)

# Experimento 3

Hipótese:
Utilizar menos atributos mantém desempenho semelhante.

In [ ]:
features_exp3 = ['Gender', 'Dependents', 'Property_Area']
X_exp3 = X[features_exp3]

# Split the data for Experiment 3
X_train_exp3, X_test_exp3, y_train_exp3, y_test_exp3 = train_test_split(
    X_exp3, y, test_size=0.20, random_state=42
)

# Train a RandomForestClassifier for Experiment 3
modelo_exp3 = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_exp3.fit(X_train_exp3, y_train_exp3)

# Make predictions
pred_exp3 = modelo_exp3.predict(X_test_exp3)

# Evaluate the model
accuracy_exp3 = accuracy_score(y_test_exp3, pred_exp3)
precision_exp3 = precision_score(y_test_exp3, pred_exp3)
recall_exp3 = recall_score(y_test_exp3, pred_exp3)
f1_exp3 = f1_score(y_test_exp3, pred_exp3)

print("Acurácia (Experimento 3):", accuracy_exp3)
print("Precisão (Experimento 3):", precision_exp3)
print("Recall (Experimento 3):", recall_exp3)
print("F1 (Experimento 3):", f1_exp3)

In [ ]:
resultados = pd.DataFrame({
    "Experimento":["Modelo Base", "Experimento 3", "Experimento 1", "Experimento 2"],
    "Acurácia":[accuracy_score(y_test,pred), accuracy_exp3, accuracy_exp1, accuracy_exp2],
    "Precisão":[precision_score(y_test,pred), precision_exp3, precision_exp1, precision_exp2],
    "Recall":[recall_score(y_test,pred), recall_exp3, recall_exp1, recall_exp2],
    "F1":[f1_score(y_test,pred), f1_exp3, f1_exp1, f1_exp2]
})

resultados

## Conclusão

O projeto AURA demonstrou que algoritmos de aprendizado de máquina podem auxiliar no processo de aprovação de empréstimos bancários.

Os experimentos indicaram que o atributo Credit_History possui maior influência na decisão de aprovação. Entre os modelos avaliados, o Random Forest apresentou o melhor equilíbrio entre precisão e recall.

A engenharia de atributos, especialmente a criação da variável TotalIncome, contribuiu para melhorar a qualidade das previsões.

Apesar dos resultados satisfatórios, o conjunto de dados possui tamanho reduzido e apresenta desbalanceamento entre as classes, o que limita a capacidade de generalização do modelo.